In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [ ]:
# Load the CSV File
df = pd.read_csv('movie.csv')

# Check the first few rows
print(df.head())

                                                text  label
0  I grew up (b. 1965) watching and loving the Th...      0
1  When I put this movie in my DVD player, and sa...      0
2  Why do people who do not know what a particula...      0
3  Even though I have great interest in Biblical ...      0
4  Im a die hard Dads Army fan and nothing will e...      1


In [ ]:
#  Split Data into Train/Validation Sets
X = df['text']
y = df['label']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [4]:
# Tokenize Text
tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

# Convert text to sequences
train_sequences = tokenizer.texts_to_sequences(X_train)
val_sequences = tokenizer.texts_to_sequences(X_val)

In [5]:
# Pad Sequences to Fixed Length
max_length = 150  # Adjust based on average review length
train_padded = pad_sequences(train_sequences, maxlen=max_length, padding='post', truncating='post')
val_padded = pad_sequences(val_sequences, maxlen=max_length, padding='post', truncating='post')

In [6]:
# Create Embedding Matrix
embeddings_index = {}
with open('glove.6B.100d.txt', encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = coefs

# Initialize embedding matrix
vocab_size = len(tokenizer.word_index) + 1
embedding_dim = 100
embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, i in tokenizer.word_index.items():
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector

In [7]:
# Build the LSTM Model
model = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        weights=[embedding_matrix],
        input_length=max_length,
        trainable=False  # Freeze embeddings (set to True to fine-tune)
    ),
    LSTM(128, dropout=0.2, recurrent_dropout=0.2),  # Increased LSTM units
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

C:\Users\david\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\keras\src\layers\core\embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │    10,143,700 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,143,700 (38.70 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 10,143,700 (38.70 MB)

In [8]:
# Train the Model
history = model.fit(
    train_padded,
    y_train,
    epochs=10,
    validation_data=(val_padded, y_val),
    batch_size=64
)

Epoch 1/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 67s 124ms/step - accuracy: 0.5581 - loss: 0.6829 - val_accuracy: 0.6780 - val_loss: 0.6097
Epoch 2/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 61s 122ms/step - accuracy: 0.6690 - loss: 0.6180 - val_accuracy: 0.7116 - val_loss: 0.5645
Epoch 3/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 69s 139ms/step - accuracy: 0.7244 - loss: 0.5544 - val_accuracy: 0.7887 - val_loss: 0.4548
Epoch 4/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 67s 133ms/step - accuracy: 0.7799 - loss: 0.4640 - val_accuracy: 0.8105 - val_loss: 0.4115
Epoch 5/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 66s 132ms/step - accuracy: 0.7998 - loss: 0.4317 - val_accuracy: 0.8180 - val_loss: 0.3995
Epoch 6/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 64s 129ms/step - accuracy: 0.8115 - loss: 0.4124 - val_accuracy: 0.8230 - val_loss: 0.3900
Epoch 7/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 65s 130ms/step - accuracy: 0.8130 - loss: 0.4062 - val_accuracy: 0.8315 - val_loss: 0.3773
Epoch 8/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 65s 131ms/step - accuracy: 0.8210 - loss: 0

In [9]:
# Predict on validation set
y_pred = (model.predict(val_padded) > 0.5).astype("int32")

# Classification report
print(classification_report(y_val, y_pred))

250/250 ━━━━━━━━━━━━━━━━━━━━ 8s 31ms/step
              precision    recall  f1-score   support

           0       0.86      0.81      0.83      3966
           1       0.82      0.87      0.84      4034

    accuracy                           0.84      8000
   macro avg       0.84      0.84      0.84      8000
weighted avg       0.84      0.84      0.84      8000



In [10]:
# Save the Model
model.save("lstm_glove_sentiment_model.h5")